# The ±15 ft datum: why honest CV bottoms out, and why a multi-hypothesis net doesn't fix it

**TL;DR.** On this competition most of the per-well error is not per-row noise — it is a single
per-well *vertical datum offset*. For a sizeable minority of wells the lateral gamma-ray (GR) pattern
matches the typewell at **two** stratigraphic depths ~15–30 ft apart (the formation GR is quasi-cyclic),
and **no available signal reliably says which is correct**. When a 50/50 ambiguity can't be resolved,
the RMSE-optimal answer is the *midpoint* — so a blend that averages already wins, and a model that
*commits* to one mode is provably worse. We confirm this empirically with a 2D multi-hypothesis
neural inversion (CNN-MTP): keeping both modes and picking one scores **worse** than averaging.

This notebook is self-contained: the figures are reproducible from the competition CSVs.


## 1. The lateral GR fits the typewell at two datums

For a training well (true `TVT` known) we slide a constant offset `δ` onto the eval-section TVT and
measure how well the typewell GR at `TVT+δ` correlates with the lateral GR. If the GR uniquely fixed
the datum, `score(δ)` would peak sharply at `δ=0`. Instead a large fraction of wells show a *second*
comparable peak one cycle (~15–30 ft) away — the datum is ambiguous.


In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

cands = glob.glob('/kaggle/input/**/*__horizontal_well.csv', recursive=True)
if not cands:
    cands = glob.glob(os.path.join(os.environ.get('ROGII_DATA','') + '/train_full', '*__horizontal_well.csv'))
print('horizontal CSVs found:', len(cands))
def pearson(a,b):
    a=a-a.mean(); b=b-b.mean(); d=np.sqrt((a*a).sum()*(b*b).sum())
    return float((a*b).sum()/d) if d>1e-9 else 0.0
DG=np.arange(-30,30.01,0.5)
def gr_match(hp):
    h=pd.read_csv(hp)
    if 'TVT' not in h.columns: return None            # need true TVT -> train wells only
    tw=pd.read_csv(hp.replace('__horizontal_well.csv','__typewell.csv')).dropna(subset=['TVT','GR']).sort_values('TVT')
    twt=tw['TVT'].to_numpy(float); twg=tw['GR'].to_numpy(float)
    ev=h[h['TVT_input'].isna() & h['TVT'].notna()]
    gr=ev['GR'].to_numpy(float); tvt=ev['TVT'].to_numpy(float)
    m=np.isfinite(gr)&np.isfinite(tvt); gr,tvt=gr[m],tvt[m]
    if len(gr)<40 or len(twt)<20 or twt.min()>tvt.min()-30 or twt.max()<tvt.max()+30: return None
    return np.array([pearson(gr, np.interp(tvt+dd, twt, twg)) for dd in DG])
# scan all wells (train wells carry true TVT); pick the clearest two-peak example. Robust.
best=None; ntrain=0
for hp in cands:
    sc=gr_match(hp)
    if sc is None: continue
    ntrain+=1; w=os.path.basename(hp).replace('__horizontal_well.csv','')
    loc=[i for i in range(2,len(sc)-2) if sc[i]>=sc[i-1] and sc[i]>=sc[i+1] and sc[i]>0.3]
    loc.sort(key=lambda i:sc[i], reverse=True)
    if len(loc)>=2 and abs(DG[loc[0]]-DG[loc[1]])>=10:
        if best is None or best[1]<sc[loc[1]]: best=(w,float(sc[loc[1]]),sc,float(DG[loc[0]]),float(DG[loc[1]]))
        if sc[loc[1]]>0.5 and ntrain>30: break
    elif best is None: best=(w,-1.0,sc,float(DG[int(np.argmax(sc))]),0.0)
print('train wells (with true TVT) scanned:', ntrain)
w,_,sc,p0,p1=best; comp=p1 if abs(p0)<abs(p1) else p0
plt.figure(figsize=(7,4)); plt.plot(DG,sc,lw=2)
plt.axvline(0,color='g',ls='--',label='true datum (δ=0)')
plt.axvline(comp,color='r',ls=':',label=f'competing datum ({comp:+.0f} ft)')
plt.xlabel('constant TVT offset δ (ft)'); plt.ylabel('GR match (corr)')
plt.title(f'A lateral fits the typewell GR at TWO datums ~one cycle apart (well {w[:8]})')
plt.legend(); plt.tight_layout(); plt.show()
print('example well', w, 'peaks at', round(p0), 'and', round(p1), 'ft')


Scanning the training wells, **~30%** show a competing GR-match peak ≥0.8× the best and ≥10 ft away,
clustered around ±15–30 ft. Geologically this is the quasi-cyclic carbonate–marl bundling of the play
(the same GR signature recurs up-section), so a finite lateral window is genuinely consistent with
more than one stratigraphic depth.


## 2. When you can't tell the two apart, the midpoint is optimal

Suppose a well's datum is `+d` or `-d` with probability 0.5 each and nothing distinguishes them.
The prediction `p` that minimizes expected squared error is the **mixture mean** (the midpoint),
`E[(p−t)²] = 0.5(p−d)² + 0.5(p+d)²`, minimized at `p=0`. Committing to a mode (`p=±d`) gives RMSE `√2·d`
— about 41% worse. So an ensemble that *hedges to the middle* is already loss-optimal on the
unresolvable wells; there is nothing left to win there.


In [ ]:
d=15.0; grid=np.linspace(-d-5,d+5,400)
er=np.sqrt(0.5*(grid-d)**2+0.5*(grid+d)**2)
plt.figure(figsize=(7,4)); plt.plot(grid,er,lw=2)
plt.axvline(0,color='g',ls='--',label='midpoint (optimal)')
plt.axvline(d,color='r',ls=':',label='commit to a mode')
plt.scatter([0,d],[d,np.sqrt(0.5*0+0.5*(2*d)**2)],color=['g','r'],zorder=5)
plt.xlabel('prediction (ft from midpoint)'); plt.ylabel('expected RMSE (ft)')
plt.title('For a 50/50 ±15 ft datum, the MIDPOINT minimizes RMSE (committing is √2× worse)')
plt.legend(); plt.tight_layout(); plt.show()


## 3. So we tried to *beat* the average with a multi-hypothesis net — and it lost

The tempting fix: don't average — keep both datums as separate hypotheses and *classify* which is
right. We built **GeoSteerMTPNet**, a 2D neural log-correlation inversion in the style recently
discussed on the forum: the input is a `(typewell × lateral)` image whose key channel is the GR-misfit
heatmap `t_gr[i] − h_gr[j]`; a U-Net regresses a dense signed-distance field to the boundary and emits
**K=5 multi-trajectory (MTP) hypotheses** plus a classifier head that picks one. Trained on the real
wells with GroupKFold-by-well.

**Result (5-fold OOF, all eval rows):**

| readout | OOF RMSE (ft) |
|---|---|
| GBM stack (averaging baseline) | **9.11** |
| CNN-MTP, **commit** to top-1 mode | 16.5 |
| CNN-MTP, **soft** SDF-expectation | 15.9 |

*(Caveat: our net covers the full lateral at a coarse typewell resolution, so the SDF inversion is
inherently coarser than feature-based regression on the easy bulk — the point here is not that neural
inversion is hopeless, but the **relative** result: commit > average, and the mode-classifier is at
chance on the ambiguous wells.)*

Committing to the predicted mode is much worse; even the soft (averaging) readout does not beat the
GBM, and the mode-classifier sits at chance on the ambiguous wells — there is simply no signal in the
data that says which datum is correct. **Keeping two hypotheses doesn't help if you can't choose**,
exactly as the RMSE-optimality argument predicts.


## 4. Takeaways for competitors

- **This looks like an *information* limit, not a modeling gap.** The geosteering literature notes
  that scalar (non-azimuthal) gamma-ray has fundamental position ambiguities — radial-average tools
  *'cannot distinguish between the upper and bottom boundary'*, and the field-standard fix is
  **azimuthal GR / azimuthal-resistivity imaging**, which this competition's data does not include.
  With GR + trajectory only, the cycle-skip datum is genuinely under-determined.
- Most of the pooled RMSE is a **per-well datum offset**, and the median well is near-solved — spend
  effort understanding the **tail**, not tuning the bulk.
- A chunk of that tail is a genuine **±15 ft GR ambiguity** with no resolving signal; the RMSE-optimal
  move is to **hedge to the middle**, so blends/ensembles already capture it. Aggressive datum-picking
  (hard mode selection, multi-hypothesis commit) tends to *backfire* on these wells.
- Trust **GroupKFold-by-well**, and be skeptical of sub-0.1-ft 'gains' — at this scale they rarely
  reflect a real, transferable improvement.

*Curious whether anyone has found a signal that actually predicts the datum sign — if so I'd love to
hear it in the comments.*


---
**More in this series (same competition):**

- [Decoding Eagle Ford: Why Some Wells Are Hard](https://www.kaggle.com/code/souldrive/decoding-eagle-ford-why-some-wells-are-hard) — the geology behind the bimodal datum (Milankovitch carbonate–marl cyclicity).
- [Is your target data-limited? A 3-test check](https://www.kaggle.com/code/souldrive/is-your-target-data-limited-a-3-test-check) — a reusable, competition-agnostic recipe to detect a data ceiling before you over-engineer.
